# Notebook 3 — Graphe OSM complet + sources externes

## Où on en est

À la fin du Notebook 2 :
- 149 traces club map-matchées
- ~12-14k arêtes OSM identifiées, mais **sans géométrie** (juste des `way_id`
  et quelques tags)
- Une heatmap "club" : nb de passages par arête

**Le problème** : tant que notre graphe `osm_edges` ne contient que les arêtes
empruntées par le club, on ne peut pas router **en dehors** de ces zones. Une
requête A→B qui passerait par une route jamais empruntée par le club
échouerait. Or notre objectif est de router partout en IdF.

## Ce qu'on fait dans ce notebook

1. **Téléchargement du graphe OSM IdF complet** via `pyrosm` (lecture du
   `.osm.pbf` qu'on a déjà)
2. **Import complet** dans PostGIS avec géométries `LINESTRING` + tags
3. **Fusion avec les données existantes** : on enrichit les arêtes déjà
   taguées "club" sans casser les liaisons `trace_edges`
4. **Téléchargement du BNAC** (Base Nationale des Aménagements Cyclables,
   data.gouv.fr) pour enrichir avec les pistes cyclables officielles
5. **Strava Segments via OAuth** : crawl de l'IdF par bbox pour récupérer
   les segments vélo populaires
6. **Calcul du score composite par arête** :
   - `score_club` : heatmap des traces club (déjà calculé)
   - `score_osm` : qualité dérivée des tags OSM
   - `score_strava` : popularité Strava mesurée
   - `score_final` : combinaison pondérée

## Pourquoi pyrosm et pas osmnx

**pyrosm** lit directement le `.osm.pbf` local → rapide (quelques minutes pour
l'IdF). **osmnx** utilise l'Overpass API publique → rate-limité et très lent
pour une région entière. On utilise pyrosm pour le bulk, osmnx éventuellement
pour des compléments locaux.

## Sources de données utilisées dans ce notebook

| Source | Volume IdF | Type | Légalité |
|---|---|---|---|
| OSM tags riches | ~500k arêtes | open data ODbL | ✓ libre |
| IdF Mobilités aménagements vélo | ~30k segments | open data | ✓ libre |
| BNAC (Base Nationale Aménagements Cyclables) | ~15k segments IdF | open data | ✓ libre |
| Strava Segments API | ~2-5k segments IdF | API OAuth officielle | ✓ légal |

## Papiers de référence pour le score composite

- **Broach et al. (2012)**. *Where do cyclists ride? A route choice model
  developed with revealed preference GPS data*. Transportation Research Part A.
  Ils apprennent les préférences réelles des cyclistes à partir de traces GPS,
  comme on va le faire dans le Notebook 5 avec nos 149 traces.
- **Ziebart et al. (2008)**. *Navigate like a cabbie*. UbiComp. Pour l'inverse
  RL si on veut pousser plus loin.

## Plan du notebook

1. Setup (imports, DB)
2. Lecture OSM IdF complète avec pyrosm
3. Insertion des nouvelles arêtes en PostGIS (sans casser l'existant)
4. Enrichissement avec BNAC
5. OAuth Strava + crawl segments
6. Snapping Strava segments → arêtes OSM
7. Calcul des 3 scores composantes
8. Score final et visualisation heatmap enrichie
9. Rapport final

## Dépendances à installer

```bash
pip install pyrosm geopandas folium branca stravalib python-dotenv
```

**Sur Windows**, `pyrosm` a besoin de `geopandas` qui a lui-même besoin de
GDAL. Si pip coince, passer par conda :

```bash
conda install -c conda-forge geopandas pyrosm
```


## 1. Setup et connexion DB

In [1]:
# Standard
from pathlib import Path
import os
import time
import json
import warnings
warnings.filterwarnings("ignore")

# Data
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Géo
import geopandas as gpd
from shapely.geometry import LineString, Point, box as shapely_box
from shapely import wkt

# DB
from sqlalchemy import create_engine, text

# HTTP
import requests

# OSM
import osmnx as ox

# Viz
import folium
from folium.plugins import HeatMap
import branca.colormap as cm

pd.set_option("display.max_columns", 60)
print("Imports OK")

Imports OK


In [2]:
# Config DB (à adapter)
DB_CONFIG = {
    "user":     "postgres",
    "password": "4421",
    "host":     "localhost",
    "port":     5432,
    "database": "velo_club",
}
url = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
       f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(url, pool_pre_ping=True)

# Chemins
OSM_PBF = Path("C:/valhalla_data/ile-de-france-latest.osm.pbf")
DATA_DIR = Path("data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Vérifs
assert OSM_PBF.exists(), f"Fichier introuvable : {OSM_PBF}"
print(f"PBF        : {OSM_PBF} ({OSM_PBF.stat().st_size / 1e6:.0f} MB)")

with engine.connect() as conn:
    n_edges_club = conn.execute(text("SELECT COUNT(*) FROM osm_edges")).scalar()
print(f"Edges club : {n_edges_club:,}")

PBF        : C:\valhalla_data\ile-de-france-latest.osm.pbf (332 MB)
Edges club : 831,949


## 2. Lecture OSM IdF complète avec pyrosm

`pyrosm.OSM("...pbf").get_network(network_type="cycling")` fait le gros du
travail : il parse le pbf, filtre les ways cyclables, et retourne un
GeoDataFrame avec une ligne par arête.

**Ce que le filtre "cycling" inclut** (selon la doc pyrosm) :
- Toutes les routes autorisées aux vélos (primary/secondary/tertiary/residential)
- Les `cycleway=*` (pistes cyclables dédiées)
- Les `path` et `footway` avec `bicycle=yes`
- Exclut les autoroutes, les routes `bicycle=no` explicites

Compte **2-5 minutes** pour l'IdF, et ~3-5 GB de RAM pic.

In [8]:
# --- Lecture OSM IdF depuis planet_osm_line ---
# On utilise uniquement les colonnes effectivement importées par osm2pgsql.
# cycleway/maxspeed/lcn/rcn sont absents : compensés par BNAC et score_club.

t0 = time.time()

CYCLING_HIGHWAYS = (
    "primary", "primary_link", "secondary", "secondary_link",
    "tertiary", "tertiary_link", "unclassified", "residential",
    "living_street", "road", "service", "track", "path",
    "pedestrian", "footway", "bridleway", "cycleway", "busway",
)

with engine.connect() as conn:
    edges = gpd.read_postgis(
        text("""
            SELECT
                osm_id AS id,
                highway,
                name,
                bicycle,
                surface,
                tracktype,
                oneway,
                access,
                ST_Transform(way, 4326) AS geometry
            FROM planet_osm_line
            WHERE highway = ANY(:hw)
              AND (bicycle IS NULL OR bicycle != 'no')
              AND (access IS NULL OR access NOT IN ('private', 'no'))
        """),
        conn,
        params={"hw": list(CYCLING_HIGHWAYS)},
        geom_col="geometry",
        crs=4326,
    )

elapsed = time.time() - t0
print(f"Lecture OSM : {elapsed:.0f} s")
print(f"Arêtes : {len(edges):,}")
print(f"Colonnes : {edges.columns.tolist()}")
print(edges["highway"].value_counts().head(10))

Lecture OSM : 16 s
Arêtes : 831,892
Colonnes : ['id', 'highway', 'name', 'bicycle', 'surface', 'tracktype', 'oneway', 'access', 'geometry']
highway
footway         297255
service         142828
residential     136092
path             58130
track            49818
tertiary         33115
secondary        31386
unclassified     26869
primary          24221
cycleway         15921
Name: count, dtype: int64


In [9]:
# Aperçu rapide
edges.head(3)

,id,highway,name,bicycle,surface,tracktype,oneway,access,geometry
0,1301540778,secondary,NaN,NaN,asphalt,NaN,NaN,NaN,"LINESTRING (3.54512 48.60249, 3.54502 48.6025,..."
1,1002779023,track,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3.54692 48.60685, 3.54703 48.60692..."
2,793917937,track,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3.55465 48.61761, 3.5547 48.61686,..."


### Nettoyage et préparation

On normalise les colonnes, on gère les NULL, et on calcule quelques features
qui serviront pour le score_osm.

In [10]:
# --- Normalisation ---
edges_clean = edges.copy()

# Pas de maxspeed dans la table : on met None
edges_clean["maxspeed_kmh"] = None

# Pas de cycleway importé : on construit une approximation depuis highway + bicycle
# Une arête est "cycleway-like" si highway=cycleway OU bicycle=designated
edges_clean["cycleway"] = None
edges_clean.loc[edges_clean["highway"] == "cycleway", "cycleway"] = "track"
edges_clean.loc[edges_clean["bicycle"] == "designated", "cycleway"] = "track"

# Pas de lcn/rcn/ncn importé : False par défaut
edges_clean["lcn_bool"] = False
edges_clean["rcn_bool"] = False
edges_clean["ncn_bool"] = False

# oneway
edges_clean["oneway_bool"] = edges_clean["oneway"].apply(
    lambda v: str(v).lower() in ("yes", "true", "1", "-1") if pd.notna(v) else False
)

# Longueur en mètres via Lambert 93
edges_clean = edges_clean.to_crs(2154)
edges_clean["length_m"] = edges_clean.geometry.length
edges_clean = edges_clean.to_crs(4326)

# Colonnes finales — mêmes noms que la version pyrosm, pour compat du reste du notebook
edges_final = edges_clean[[
    "id", "length_m", "highway", "cycleway", "surface",
    "maxspeed_kmh", "bicycle",
    "lcn_bool", "rcn_bool", "ncn_bool", "oneway_bool",
    "geometry",
]].copy()

# Filtre des géométries invalides
edges_final = edges_final[edges_final.geometry.is_valid].copy()

print(f"{len(edges_final):,} arêtes prêtes à insérer")
print(f"Distance totale : {edges_final['length_m'].sum() / 1000:,.0f} km")
print(f"Avec cycleway approximé : {edges_final['cycleway'].notna().sum():,}")
print(f"  (dont highway=cycleway : {(edges_final['highway'] == 'cycleway').sum():,})")
print(f"  (dont bicycle=designated : {(edges_final['bicycle'] == 'designated').sum():,})")

831,892 arêtes prêtes à insérer
Distance totale : 101,446 km
Avec cycleway approximé : 25,957
  (dont highway=cycleway : 15,921)
  (dont bicycle=designated : 11,191)


## 3. Insertion en base avec fusion intelligente

### Le problème

Notre table `osm_edges` contient déjà ~12k arêtes **empruntées par le club**,
référencées par `trace_edges.edge_id`. Si on TRUNCATE et on réinsère, on perd
toutes les jointures, et il faut tout refaire.

### La solution

On ajoute un champ `osm_way_id` UNIQUE (déjà prévu dans le schéma) et on fait
un **UPSERT** : si `osm_way_id` existe déjà, on met à jour les tags et la
géométrie sans changer le `edge_id`. Sinon on insère.

On fait ça par batch de 5000 pour aller vite.

In [11]:
# D'abord on ajoute la contrainte UNIQUE sur osm_way_id si elle n'y est pas
with engine.begin() as conn:
    conn.execute(text("""
        -- Ajoute la contrainte d'unicité si absente
        DO $$
        BEGIN
            IF NOT EXISTS (
                SELECT 1 FROM pg_constraint
                WHERE conname = 'osm_edges_way_id_unique'
            ) THEN
                ALTER TABLE osm_edges
                ADD CONSTRAINT osm_edges_way_id_unique UNIQUE (osm_way_id);
            END IF;
        END $$;
    """))
print("Contrainte UNIQUE sur osm_way_id OK")

Contrainte UNIQUE sur osm_way_id OK


In [12]:
# --- Stats avant insertion ---
with engine.connect() as conn:
    stats_avant = pd.read_sql(text("""
        SELECT
            COUNT(*) AS total,
            COUNT(geom) AS avec_geom,
            COUNT(*) - COUNT(geom) AS sans_geom
        FROM osm_edges;
    """), conn)
print("Avant insertion :")
print(stats_avant.to_string(index=False))

Avant insertion :
 total  avec_geom  sans_geom
 13120          0      13120


In [13]:
# --- Upsert par batch ---
# On utilise le WKT pour passer les géométries en SQL (simple, marche partout)

upsert_sql = text("""
INSERT INTO osm_edges (
    osm_way_id, length_m, highway, cycleway, surface, maxspeed, bicycle,
    lcn, rcn, oneway, geom
) VALUES (
    :osm_way_id, :length_m, :highway, :cycleway, :surface, :maxspeed, :bicycle,
    :lcn, :rcn, :oneway, ST_GeomFromText(:geom_wkt, 4326)
)
ON CONFLICT (osm_way_id) DO UPDATE SET
    length_m  = EXCLUDED.length_m,
    highway   = EXCLUDED.highway,
    cycleway  = EXCLUDED.cycleway,
    surface   = EXCLUDED.surface,
    maxspeed  = EXCLUDED.maxspeed,
    bicycle   = EXCLUDED.bicycle,
    lcn       = EXCLUDED.lcn,
    rcn       = EXCLUDED.rcn,
    oneway    = EXCLUDED.oneway,
    geom      = EXCLUDED.geom;
""")


def row_to_params(row):
    # cycleway synthétique : n'importe lequel des cycleway_* non-null positif
    cw = row.get("cycleway")
    if pd.isna(cw):
        for col in ("cycleway:left", "cycleway:right"):
            if col in row and pd.notna(row[col]):
                cw = row[col]
                break
    if pd.isna(cw):
        cw = None
    return {
        "osm_way_id": int(row["id"]),
        "length_m":   float(row["length_m"]),
        "highway":    row.get("highway") if pd.notna(row.get("highway")) else None,
        "cycleway":   str(cw) if cw is not None else None,
        "surface":    row.get("surface") if pd.notna(row.get("surface")) else None,
        "maxspeed":   int(row["maxspeed_kmh"]) if pd.notna(row.get("maxspeed_kmh")) else None,
        "bicycle":    row.get("bicycle") if pd.notna(row.get("bicycle")) else None,
        "lcn":        bool(row.get("lcn_bool", False)),
        "rcn":        bool(row.get("rcn_bool", False)),
        "oneway":     bool(row.get("oneway_bool", False)),
        "geom_wkt":   row["geometry"].wkt,
    }


# Insert par batch pour ne pas exploser la RAM
BATCH_SIZE = 5000
n_total = len(edges_final)

with engine.begin() as conn:
    for start in tqdm(range(0, n_total, BATCH_SIZE), desc="Upsert OSM"):
        batch = edges_final.iloc[start:start + BATCH_SIZE]
        params = [row_to_params(r) for _, r in batch.iterrows()]
        conn.execute(upsert_sql, params)

print("\n✓ Upsert terminé")

Upsert OSM: 100%|██████████| 167/167 [05:39<00:00,  2.03s/it]


✓ Upsert terminé


In [14]:
# --- Stats après ---
with engine.connect() as conn:
    stats_apres = pd.read_sql(text("""
        SELECT
            COUNT(*) AS total,
            COUNT(geom) AS avec_geom,
            COUNT(*) - COUNT(geom) AS sans_geom,
            COUNT(DISTINCT highway) AS classes_distinctes
        FROM osm_edges;
    """), conn)
print("Après insertion :")
print(stats_apres.to_string(index=False))

# Vérif : les trace_edges pointent toujours bien
with engine.connect() as conn:
    orphans = conn.execute(text("""
        SELECT COUNT(*) FROM trace_edges te
        LEFT JOIN osm_edges e ON e.edge_id = te.edge_id
        WHERE e.edge_id IS NULL;
    """)).scalar()
print(f"trace_edges orphelines : {orphans} (doit être 0)")

Après insertion :
 total  avec_geom  sans_geom  classes_distinctes
831949     831892         57                  19
trace_edges orphelines : 0 (doit être 0)


## 4. Enrichissement avec la Base Nationale des Aménagements Cyclables

La **BNAC** est un dataset officiel data.gouv.fr qui compile les pistes
cyclables déclarées par les collectivités françaises, avec plus d'attributs
que OSM (largeur, séparation du trafic, statut légal).

On télécharge, on filtre sur l'IdF, et on **snap** ces segments à nos arêtes
OSM : si un segment BNAC chevauche une arête OSM à > 50%, on tag cette arête
comme "piste cyclable officielle".

In [3]:
BNAC_LOCAL = Path(r"C:\Users\rapha\Desktop\GPX projet\Data\raw\france-20260410.geojson")

# Skip le téléchargement puisqu'il existe déjà
print(f"Utilisation du fichier : {BNAC_LOCAL}")
print(f"Taille : {BNAC_LOCAL.stat().st_size / 1e6:.0f} MB")

# Chargement direct
print("Chargement du geojson (compte 20-30 s)...")
t0 = time.time()
bnac = gpd.read_file(BNAC_LOCAL)
print(f"BNAC France : {len(bnac):,} segments (lu en {time.time()-t0:.0f}s)")
print(f"Colonnes    : {bnac.columns.tolist()[:15]}")
bnac.head(3)

Utilisation du fichier : C:\Users\rapha\Desktop\GPX projet\Data\raw\france-20260410.geojson
Taille : 295 MB
Chargement du geojson (compte 20-30 s)...
BNAC France : 397,326 segments (lu en 34s)
Colonnes    : ['id_local', 'id_osm', 'num_iti', 'code_com_d', 'ame_d', 'regime_d', 'sens_d', 'largeur_d', 'local_d', 'statut_d', 'revet_d', 'code_com_g', 'ame_g', 'regime_g', 'sens_g']


,id_local,id_osm,num_iti,code_com_d,ame_d,regime_d,sens_d,largeur_d,local_d,statut_d,revet_d,code_com_g,ame_g,regime_g,sens_g,largeur_g,local_g,statut_g,revet_g,access_ame,date_maj,trafic_vit,lumiere,d_service,source,project_c,ref_geo,geometry
0,geovelo_570327600_2B123,570327600,NaN,2B123,AUTRE,AUTRE,UNIDIRECTIONNEL,0.0,NaN,EN SERVICE,RUGUEUX,2B123,AUTRE,AUTRE,UNIDIRECTIONNEL,0.0,NaN,EN SERVICE,RUGUEUX,NaN,2018-03-16,5.0,NaN,NaN,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (9.47212 42.03309, 9.47244 42.03352..."
1,geovelo_621800068_2B007,621800068,NaN,2B007,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,NaN,EN SERVICE,NaN,2B007,AUTRE,AUTRE,UNIDIRECTIONNEL,NaN,NaN,EN SERVICE,NaN,VTC,2025-07-01,5.0,NaN,NaN,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (8.9074 42.27899, 8.90731 42.27907)"
2,geovelo_927735333_2A004,927735333,NaN,2A004,PISTE CYCLABLE,AUTRE,UNIDIRECTIONNEL,NaN,NaN,EN SERVICE,NaN,2A004,PISTE CYCLABLE,AUTRE,UNIDIRECTIONNEL,NaN,NaN,EN SERVICE,NaN,NaN,2023-09-10,5.0,NaN,NaN,Les contributeurs OpenStreetmap,4326,OpenStreetmap,"LINESTRING (8.74134 41.93641, 8.74134 41.93638..."


In [4]:
# Filtrage sur IdF
IDF_BBOX = shapely_box(1.2, 48.0, 3.6, 49.3)

if len(bnac):
    bnac = bnac.to_crs(4326)
    bnac_idf = bnac[bnac.geometry.intersects(IDF_BBOX)].copy()
    print(f"BNAC IdF : {len(bnac_idf):,} segments")

    # Sauvegarde locale pour réutilisation
    bnac_idf.to_file(DATA_DIR / "bnac_idf.geojson", driver="GeoJSON")
else:
    bnac_idf = gpd.GeoDataFrame()

BNAC IdF : 66,671 segments


### Snapping des segments BNAC vers les arêtes OSM

Pour chaque segment BNAC, on trouve les arêtes OSM qui se trouvent à moins de
15 m, et on les tag `has_bnac = TRUE`. On fait ça en SQL pour exploiter
l'index GIST de PostGIS.

In [ ]:
with engine.begin() as conn:
    # Crée un index spatial sur bnac_tmp si absent
    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS idx_bnac_tmp_geom
        ON bnac_tmp USING GIST (geometry);
        ANALYZE bnac_tmp;
    """))
    # Vérifie aussi que l'index sur osm_edges existe
    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS idx_osm_edges_geom
        ON osm_edges USING GIST (geom);
        ANALYZE osm_edges;
    """))
print("Index spatiaux créés")

Index spatiaux créés


In [6]:
print("Snapping optimisé (doit finir en 2-5 min)...")
t0 = time.time()

RADIUS_DEG = 15 / 111000  # ~0.000135°, couvre 15m en latitude

with engine.begin() as conn:
    updated = conn.execute(text(f"""
        UPDATE osm_edges
        SET has_bnac = TRUE
        WHERE edge_id IN (
            SELECT DISTINCT e.edge_id
            FROM osm_edges e
            JOIN bnac_tmp b
              ON ST_DWithin(e.geom, b.geometry, {RADIUS_DEG})
             AND ST_DWithin(e.geom::geography, b.geometry::geography, 15)
        );
    """)).rowcount

print(f"\n✓ {updated:,} arêtes OSM taggées BNAC en {time.time()-t0:.0f}s")

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS bnac_tmp;"))
print("bnac_tmp supprimé")

Snapping optimisé (doit finir en 2-5 min)...

✓ 285,313 arêtes OSM taggées BNAC en 118s
bnac_tmp supprimé


## 5. Strava Segments via OAuth officiel

### Principe

Strava expose un endpoint `/segments/explore` qui retourne les **10 segments
les plus populaires** dans une bounding box. L'endpoint est
https://www.strava.com/api/v3/segments/explore?bounds=... et il est
**limité à 10 segments par requête** quelle que soit la taille de la bbox.

Stratégie : **quadriller l'IdF en petites bbox** (grille de ~5×5 km), une
requête par cellule → on récupère ~2-5k segments uniques au total.

### Rate limiting

Le rate limit par défaut autorise 200 requêtes toutes les 15 minutes, avec
jusqu'à 2000 requêtes par jour. Pour une grille 40×25 = 1000 cellules,
on peut tout crawler en ~1 journée en respectant le rate limit.

### Étape 1 : création de ton app Strava

Va sur https://www.strava.com/settings/api et crée une application :
- Name : ton projet
- Category : "training"
- Authorization Callback Domain : `localhost`

Tu obtiens :
- `Client ID`
- `Client Secret`

### Étape 2 : OAuth manuel (one-shot)

Pour récupérer un `access_token` avec le scope `read,read_all`, suis ce
flow une fois, et stocke le refresh_token en fichier `.env`.

In [3]:
# --- Config Strava ---
# Crée un fichier .env à la racine avec :
#   STRAVA_CLIENT_ID=xxx
#   STRAVA_CLIENT_SECRET=xxx
#   STRAVA_REFRESH_TOKEN=xxx
# Et charge-le ici avec python-dotenv.

from dotenv import load_dotenv
load_dotenv()

STRAVA_CLIENT_ID     = os.getenv("STRAVA_CLIENT_ID")
STRAVA_CLIENT_SECRET = os.getenv("STRAVA_CLIENT_SECRET")
STRAVA_REFRESH_TOKEN = os.getenv("STRAVA_REFRESH_TOKEN")

if not all([STRAVA_CLIENT_ID, STRAVA_CLIENT_SECRET, STRAVA_REFRESH_TOKEN]):
    print("""
    ⚠ Credentials Strava manquants. Fais ceci :

    1. Crée ton app sur https://www.strava.com/settings/api
    2. Récupère un code d'autorisation en ouvrant dans ton navigateur :
       https://www.strava.com/oauth/authorize?client_id=TON_CLIENT_ID&redirect_uri=http://localhost&response_type=code&scope=read,read_all

    3. Une fois accepté, tu es redirigé vers localhost avec un code=XXX
       dans l'URL. Copie ce code.

    4. Échange le code contre un refresh_token (dans un terminal) :
       curl -X POST https://www.strava.com/oauth/token \
         -F client_id=TON_CLIENT_ID \
         -F client_secret=TON_CLIENT_SECRET \
         -F code=LE_CODE_COPIE \
         -F grant_type=authorization_code

    5. Dans la réponse JSON, récupère refresh_token et stocke-le dans .env
    """)
else:
    print("✓ Credentials Strava trouvés")

✓ Credentials Strava trouvés


In [4]:
def strava_access_token():
    """
    Échange le refresh_token contre un access_token frais.
    Les access_token Strava expirent après 6 heures ; on regénère à la demande.
    """
    r = requests.post("https://www.strava.com/oauth/token", data={
        "client_id":     STRAVA_CLIENT_ID,
        "client_secret": STRAVA_CLIENT_SECRET,
        "refresh_token": STRAVA_REFRESH_TOKEN,
        "grant_type":    "refresh_token",
    }, timeout=30)
    r.raise_for_status()
    return r.json()["access_token"]


def explore_segments(bounds, access_token):
    """
    Appel /segments/explore pour une bbox.
    bounds = (sw_lat, sw_lon, ne_lat, ne_lon)
    Retourne jusqu'à 10 segments.
    """
    r = requests.get(
        "https://www.strava.com/api/v3/segments/explore",
        params={
            "bounds":        ",".join(map(str, bounds)),
            "activity_type": "riding",
        },
        headers={"Authorization": f"Bearer {access_token}"},
        timeout=30,
    )
    # Gestion rate limit
    if r.status_code == 429:
        # Attendre jusqu'au prochain quart d'heure
        print("⏳ Rate limit, attente 15 min…")
        time.sleep(15 * 60)
        return explore_segments(bounds, access_token)
    r.raise_for_status()
    return r.json().get("segments", [])


# # Test
# if STRAVA_REFRESH_TOKEN:
#     token = strava_access_token()
#     test_bounds = (48.84, 2.33, 48.87, 2.36)   # petit coin de Paris
#     segs = explore_segments(test_bounds, token)
#     print(f"Test : {len(segs)} segments trouvés dans Paris centre")
#     if segs:
#         print(f"Premier : {segs[0]['name']} ({segs[0]['distance']:.0f} m)")

### Grille IdF + crawl

On découpe l'IdF en cellules de ~3 km × 3 km et on tape chacune.

In [ ]:
import json
from pathlib import Path
from shapely.geometry import Point, Polygon, box as shapely_box
from shapely.ops import unary_union

if STRAVA_REFRESH_TOKEN:
    # --- Stratégie de ciblage ---
    # On exclut :
    # 1. Paris + petite couronne dense (rayon 12 km autour de Notre-Dame)
    #    Les segments sont majoritairement du vélotaf, pas pertinents pour ton usage road
    # 2. On garde seulement les cellules qui sont à moins de 50 km de Pantin
    #    (au-delà, ton club ne sort jamais)
    
    NOTRE_DAME = (48.8530, 2.3500)   # centre Paris
    PANTIN     = (48.8922, 2.4014)   # HQ club
    EXCLUDE_PARIS_RADIUS_KM = 15
    MAX_DIST_FROM_PANTIN_KM = 100
    
    def dist_km(lat1, lon1, lat2, lon2):
        
        from math import radians, sin, cos, atan2, sqrt
        R = 6371
        phi1, phi2 = radians(lat1), radians(lat2)
        dphi = radians(lat2 - lat1)
        dlam = radians(lon2 - lon1)
        a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlam/2)**2
        return 2 * R * atan2(sqrt(a), sqrt(1 - a))
    
    def cell_utile(lat, lon, step=0.04):
        
        center_lat = lat + step / 2
        center_lon = lon + step / 2
        # 1) pas trop loin de Pantin
        d_pantin = dist_km(center_lat, center_lon, *PANTIN)
        if d_pantin > MAX_DIST_FROM_PANTIN_KM:
            return False
        # 2) hors du centre de Paris
        d_paris = dist_km(center_lat, center_lon, *NOTRE_DAME)
        if d_paris < EXCLUDE_PARIS_RADIUS_KM:
            return False
        return True
    
    # Grille + filtrage
    STEP = 0.04
    lons = np.arange(1.2, 3.6, STEP)
    lats = np.arange(48.0, 49.3, STEP)
    
    all_cells = []
    for lon in lons:
        for lat in lats:
            if cell_utile(lat, lon, STEP):
                all_cells.append((round(float(lat), 3), round(float(lon), 3)))
    
    print(f"Grille brute    : {len(lons) * len(lats)} cellules")
    print(f"Grille ciblée   : {len(all_cells)} cellules")
    print(f"Réduction       : {100 * (1 - len(all_cells) / (len(lons)*len(lats))):.0f}%")
    print(f"Temps estimé    : {len(all_cells) * 4.5 / 3600:.1f}h "
          f"(à 4.5s/req, hors pauses rate limit)")
    
    # Checkpoint reprise
    CHECKPOINT = DATA_DIR / "strava_segments_checkpoint.json"
    DONE_FILE  = DATA_DIR / "strava_cells_done.json"
    
    if CHECKPOINT.exists():
        with open(CHECKPOINT) as f:
            all_segments = {int(k): v for k, v in json.load(f).items()}
        print(f"✓ Checkpoint : {len(all_segments)} segments déjà collectés")
    else:
        all_segments = {}
    
    if DONE_FILE.exists():
        with open(DONE_FILE) as f:
            done_cells = set(tuple(c) for c in json.load(f))
    else:
        done_cells = set()
    
    remaining = [c for c in all_cells if c not in done_cells]
    print(f"  Cellules restantes : {len(remaining)}")
    print(f"  Temps restant est. : {len(remaining) * 4.5 / 3600:.1f}h")
    
    # --- Crawl ---
    token = strava_access_token()
    t0 = time.time()
    calls = 0
    SAVE_EVERY = 50
    
    def save_checkpoint():
        with open(CHECKPOINT, "w") as f:
            json.dump({str(k): v for k, v in all_segments.items()}, f)
        with open(DONE_FILE, "w") as f:
            json.dump([list(c) for c in done_cells], f)
    
    try:
        for (lat, lon) in tqdm(remaining, desc="Crawl Strava ciblé"):
            if calls > 0 and calls % 500 == 0:
                token = strava_access_token()
            
            bounds = (lat, lon, lat + STEP, lon + STEP)
            try:
                segs = explore_segments(bounds, token)
                for s in segs:
                    all_segments[s["id"]] = s
                done_cells.add((lat, lon))
            except Exception as e:
                if "429" in str(e):
                    print(f"\\n⏳ Rate limit atteint, pause 15 min...")
                    save_checkpoint()
                    time.sleep(15 * 60 + 10)
                    token = strava_access_token()
                    try:
                        segs = explore_segments(bounds, token)
                        for s in segs:
                            all_segments[s["id"]] = s
                        done_cells.add((lat, lon))
                    except Exception:
                        pass
            
            calls += 1
            if calls % SAVE_EVERY == 0:
                save_checkpoint()
            
            time.sleep(4.5)   # respecte 13 req/min
    
    except KeyboardInterrupt:
        print("\\n⚠ Interrompu, sauvegarde...")
        save_checkpoint()
        raise
    
    save_checkpoint()
    print(f"\\n✓ {len(all_segments)} segments collectés en {time.time()-t0:.0f}s")
    
    with open(DATA_DIR / "strava_segments_idf.json", "w") as f:
        json.dump(list(all_segments.values()), f)
else:
    all_segments = {}
    print("⚠ Strava non configuré, on skip")

Grille brute    : 2013 cellules
Grille ciblée   : 1753 cellules
Réduction       : 13%
Temps estimé    : 2.2h (à 4.5s/req, hors pauses rate limit)
✓ Checkpoint : 6130 segments déjà collectés
  Cellules restantes : 903
  Temps restant est. : 1.1h


Crawl Strava ciblé:  21%|██▏       | 193/903 [23:05<1:07:17,  5.69s/it]

⏳ Rate limit, attente 15 min…


### Insertion en DB

In [7]:
if all_segments:
    # Vider la table (idempotent)
    with engine.begin() as conn:
        conn.execute(text("TRUNCATE TABLE strava_segments RESTART IDENTITY;"))

    insert_seg_sql = text("""
    INSERT INTO strava_segments (
        segment_id, name, activity_type, distance_m, average_grade,
        effort_count, star_count, geom
    ) VALUES (
        :id, :name, :activity_type, :distance, :average_grade,
        :effort_count, :star_count, ST_GeomFromText(:geom_wkt, 4326)
    )
    ON CONFLICT (segment_id) DO NOTHING;
    """)

    def decode_polyline5(encoded):
        """Décode la polyline Strava (precision 1e5)."""
        coords = []
        index = lat = lng = 0
        while index < len(encoded):
            shift = result = 0
            while True:
                b = ord(encoded[index]) - 63
                index += 1
                result |= (b & 0x1f) << shift
                shift += 5
                if b < 0x20:
                    break
            dlat = ~(result >> 1) if result & 1 else (result >> 1)
            lat += dlat
            shift = result = 0
            while True:
                b = ord(encoded[index]) - 63
                index += 1
                result |= (b & 0x1f) << shift
                shift += 5
                if b < 0x20:
                    break
            dlng = ~(result >> 1) if result & 1 else (result >> 1)
            lng += dlng
            coords.append((lng / 1e5, lat / 1e5))
        return coords

    inserted = 0
    with engine.begin() as conn:
        for seg in tqdm(all_segments.values(), desc="Insert Strava"):
            coords = decode_polyline5(seg.get("points", ""))
            if len(coords) < 2:
                continue
            ls = LineString(coords)
            try:
                conn.execute(insert_seg_sql, {
                    "id":             int(seg["id"]),
                    "name":           seg.get("name"),
                    "activity_type":  seg.get("activity_type", "Ride"),
                    "distance":       float(seg.get("distance", 0)),
                    "average_grade":  float(seg.get("avg_grade", 0)),
                    "effort_count":   int(seg.get("effort_count", 0) or 0),
                    "star_count":     int(seg.get("star_count", 0) or 0),
                    "geom_wkt":       ls.wkt,
                })
                inserted += 1
            except Exception:
                pass

    print(f"✓ {inserted} segments Strava insérés")

Insert Strava: 100%|██████████| 6130/6130 [00:02<00:00, 2554.65it/s]

✓ 6130 segments Strava insérés


## 5. Features géographiques : densité urbaine et zones vertes

Avant de passer au scoring (Notebook 3bis), on enrichit `osm_edges` avec deux
features qu'on va utiliser dans la fonction de coût :

### 5.1 Densité de nœuds routiers

**Intuition** : un résidentiel de village est sympa à vélo, un résidentiel en
centre-ville parisien est chiant (stops, voitures, piétons). OSM ne tague pas
"village vs ville", mais on peut le **proxy-er** via la **densité d'intersections**
du réseau routier : une zone dense a beaucoup de petites rues entrecroisées,
une zone rurale en a peu.

**Méthode** : pour chaque arête, on compte le nombre de nœuds du graphe routier
dans un buffer de 500m autour de son centroïde.

- Densité < 50 nœuds → village/campagne → bonus
- Densité 50-200 → pavillonnaire/petite ville → neutre
- Densité > 200 → urbain dense → malus

### 5.2 Proximité zones vertes

**Intuition** : rouler en forêt/près de champs = agréable. OSM tague ça avec
`landuse=forest`, `landuse=farmland`, `leisure=park`, `natural=wood`.

**Méthode** : pour chaque arête, on calcule la fraction de sa longueur qui
est à moins de 100m d'une zone verte OSM.

Ces deux features seront stockées dans `osm_edges` pour servir au scoring.

In [3]:
# Ajout des colonnes
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE osm_edges ADD COLUMN IF NOT EXISTS node_density INTEGER;
        ALTER TABLE osm_edges ADD COLUMN IF NOT EXISTS greenery_ratio DOUBLE PRECISION;
    """))
print("Colonnes node_density et greenery_ratio ajoutées.")

Colonnes node_density et greenery_ratio ajoutées.


### 5.1 Densité de nœuds du graphe routier

In [4]:
# --- Construction d'une table de nœuds uniques (centroids virtuels des intersections) ---
# On utilise ST_StartPoint + ST_EndPoint de chaque arête, dédupliqués

print("Construction des nœuds routiers...")
import time
t0 = time.time()

with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS osm_nodes_tmp;
        CREATE TABLE osm_nodes_tmp AS
        SELECT DISTINCT ST_SnapToGrid(geom, 0.00001) AS geom
        FROM (
            SELECT ST_StartPoint(geom) AS geom FROM osm_edges
            UNION ALL
            SELECT ST_EndPoint(geom) FROM osm_edges
        ) sub;
        
        CREATE INDEX idx_nodes_tmp_geom ON osm_nodes_tmp USING GIST (geom);
        ANALYZE osm_nodes_tmp;
    """))

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM osm_nodes_tmp")).scalar()
print(f"  {n:,} nœuds uniques en {time.time()-t0:.0f}s")

Construction des nœuds routiers...
  1,098,740 nœuds uniques en 11s


In [5]:
import time
from sqlalchemy import text

print("Étape 1 : Nettoyage et création des index spatiaux (Lambert 93)...")
t0 = time.time()

with engine.begin() as conn:
    # 1. Nettoyage : On supprime les colonnes si elles existent déjà pour repartir au propre
    conn.execute(text("""
        ALTER TABLE osm_nodes_tmp DROP COLUMN IF EXISTS geom_2154;
        ALTER TABLE osm_edges DROP COLUMN IF EXISTS centroid_2154;
    """))

    # 2. Création des nouvelles colonnes
    conn.execute(text("""
        ALTER TABLE osm_nodes_tmp ADD COLUMN geom_2154 geometry(Point, 2154);
        ALTER TABLE osm_edges ADD COLUMN centroid_2154 geometry(Point, 2154);
    """))

    # 3. Remplissage des colonnes avec la bonne projection
    print(" -> Transformation des géométries en cours...")
    conn.execute(text("""
        UPDATE osm_nodes_tmp SET geom_2154 = ST_Transform(geom, 2154);
        UPDATE osm_edges SET centroid_2154 = ST_Transform(ST_Centroid(geom), 2154);
    """))

    # 4. Création des index spatiaux (c'est ça qui rend la suite ultra-rapide)
    print(" -> Création des index spatiaux GIST...")
    conn.execute(text("""
        CREATE INDEX idx_osm_nodes_geom2154 ON osm_nodes_tmp USING GIST (geom_2154);
        CREATE INDEX idx_osm_edges_centroid2154 ON osm_edges USING GIST (centroid_2154);
    """))

    # 5. Mise à jour des statistiques de l'optimiseur PostgreSQL
    conn.execute(text("ANALYZE osm_nodes_tmp; ANALYZE osm_edges;"))

print(f"✅ Préparation terminée en {time.time()-t0:.0f}s")

Étape 1 : Nettoyage et création des index spatiaux (Lambert 93)...
 -> Transformation des géométries en cours...
 -> Création des index spatiaux GIST...
✅ Préparation terminée en 104s


In [6]:
import time
import pandas as pd
from sqlalchemy import text

print("Étape 2 : Calcul de la densité de nœuds...")
t0 = time.time()

with engine.begin() as conn:
    # Requête de calcul optimisée utilisant ST_DWithin sur les index
    conn.execute(text("""
        WITH counts AS (
            SELECT e.edge_id, COUNT(n.*) AS n_nodes
            FROM osm_edges e
            LEFT JOIN osm_nodes_tmp n
              ON ST_DWithin(e.centroid_2154, n.geom_2154, 500)
            GROUP BY e.edge_id
        )
        UPDATE osm_edges e
        SET node_density = c.n_nodes
        FROM counts c
        WHERE e.edge_id = c.edge_id;
    """))

print(f"✅ Calcul terminé en {time.time()-t0:.0f}s")

# Récupération et affichage des statistiques
print("\n📊 Récupération des statistiques...")
with engine.connect() as conn:
    stats = pd.read_sql(text("""
        SELECT
            ROUND(AVG(node_density)::numeric, 1) AS mean,
            MIN(node_density) AS min,
            percentile_cont(0.5) WITHIN GROUP (ORDER BY node_density) AS median,
            percentile_cont(0.9) WITHIN GROUP (ORDER BY node_density) AS p90,
            MAX(node_density) AS max
        FROM osm_edges;
    """), conn)

print(stats.to_string(index=False))

Étape 2 : Calcul de la densité de nœuds...
✅ Calcul terminé en 689s

📊 Récupération des statistiques...
 mean  min  median    p90  max
464.6    0   318.0 1155.0 2660


In [7]:
# Nettoyage
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS osm_nodes_tmp;"))
print("osm_nodes_tmp supprimé.")

osm_nodes_tmp supprimé.


### 5.2 Zones vertes OSM (landuse)

On requête `planet_osm_polygon` pour récupérer les polygones de type forêt,
parc, champ, bois, etc. qu'`osm2pgsql` a importés. Puis pour chaque arête on
calcule quelle fraction est à moins de 100m d'une zone verte.

In [3]:
# --- Extraction des zones vertes depuis planet_osm_polygon ---
# Les tags qui nous intéressent : landuse/natural/leisure pour les espaces naturels

print("Extraction des zones vertes...")
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS green_zones_tmp;
        CREATE TABLE green_zones_tmp AS
        SELECT ST_Transform(way, 2154) AS geom
        FROM planet_osm_polygon
        WHERE landuse IN ('forest', 'farmland', 'meadow', 'vineyard', 'orchard',
                          'grass', 'recreation_ground', 'village_green')
           OR "natural" IN ('wood', 'grassland', 'scrub', 'heath')
           OR leisure IN ('park', 'nature_reserve', 'garden');
        
        CREATE INDEX idx_green_tmp_geom ON green_zones_tmp USING GIST (geom);
        ANALYZE green_zones_tmp;
    """))

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM green_zones_tmp")).scalar()
    total_km2 = conn.execute(text("""
        SELECT ROUND((SUM(ST_Area(geom))/1e6)::numeric, 0)
        FROM green_zones_tmp;
    """)).scalar()
print(f"  {n:,} polygones de zones vertes, {total_km2} km² au total")

Extraction des zones vertes...
  128,903 polygones de zones vertes, 11791 km² au total


In [4]:
import time
from sqlalchemy import text

print("Étape 1 : Préparation des données (Lambert 93 + Buffer 100m)...")
t0 = time.time()

with engine.begin() as conn:
    # 1. Vérification/Création de la ligne complète en Lambert 93 pour les arêtes
    # (Si tu ne l'as pas déjà fait dans tes étapes précédentes)
    conn.execute(text("ALTER TABLE osm_edges ADD COLUMN IF NOT EXISTS geom_2154 geometry(LineString, 2154);"))
    conn.execute(text("UPDATE osm_edges SET geom_2154 = ST_Transform(geom, 2154) WHERE geom_2154 IS NULL;"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_osm_edges_geom2154 ON osm_edges USING GIST (geom_2154);"))

    # 2. Création des zones vertes AVEC le buffer de 100m pré-calculé
    # Puisque ta table green_zones_tmp est DEJA en 2154, on applique juste le buffer
    conn.execute(text("DROP TABLE IF EXISTS green_zones_buf;"))
    conn.execute(text("""
        CREATE TABLE green_zones_buf AS
        SELECT ST_Buffer(geom, 100) AS geom_buf
        FROM green_zones_tmp;
    """))

    # 3. Indexation spatiale du buffer (L'étape magique pour la vitesse)
    conn.execute(text("CREATE INDEX idx_green_zones_buf ON green_zones_buf USING GIST (geom_buf);"))

    # 4. Mise à jour des statistiques
    conn.execute(text("ANALYZE osm_edges; ANALYZE green_zones_buf;"))

print(f"✅ Préparation terminée en {time.time()-t0:.0f}s")

Étape 1 : Préparation des données (Lambert 93 + Buffer 100m)...
✅ Préparation terminée en 84s


In [5]:
import time
import pandas as pd
from sqlalchemy import text

print("Étape 2 : Calcul du greenery_ratio...")
t0 = time.time()

with engine.begin() as conn:
    conn.execute(text("""
        WITH green_buf AS (
            SELECT e.edge_id,
                COALESCE(
                    SUM(
                        -- On calcule la longueur de l'arête qui est contenue dans le buffer pré-calculé
                        ST_Length(ST_Intersection(e.geom_2154, g.geom_buf))
                    ), 0
                ) AS green_len
            FROM osm_edges e
            -- Jointure spatiale foudroyante grâce aux index
            LEFT JOIN green_zones_buf g
              ON ST_Intersects(e.geom_2154, g.geom_buf)
            GROUP BY e.edge_id
        )
        UPDATE osm_edges oe
        SET greenery_ratio = LEAST(1.0, gb.green_len / NULLIF(oe.length_m, 0))
        FROM green_buf gb
        WHERE oe.edge_id = gb.edge_id;
    """))

print(f"✅ Calcul terminé en {time.time()-t0:.0f}s")

# Récupération et affichage des statistiques
print("\n📊 Récupération des statistiques...")
with engine.connect() as conn:
    stats = pd.read_sql(text("""
        SELECT
            COUNT(*) FILTER (WHERE greenery_ratio > 0) AS avec_vert,
            COUNT(*) FILTER (WHERE greenery_ratio > 0.5) AS majoritaire_vert,
            ROUND(AVG(greenery_ratio)::numeric, 3) AS mean
        FROM osm_edges;
    """), conn)

print(stats.to_string(index=False))

Étape 2 : Calcul du greenery_ratio...
✅ Calcul terminé en 497s

📊 Récupération des statistiques...
 avec_vert  majoritaire_vert  mean
    662609            628046 0.754


In [6]:
# Nettoyage
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS green_zones_tmp;"))
print("green_zones_tmp supprimé.")

green_zones_tmp supprimé.


### Vérification finale

On a maintenant dans `osm_edges` :
- `length_m`, `highway`, `surface`, `bicycle`, `oneway`, `maxspeed` (OSM)
- `has_bnac` (aménagements cyclables officiels)
- `node_density` (proxy ville vs village)
- `greenery_ratio` (proximité aux zones vertes)
- `d_plus_m`, `avg_grade`, `alt_std` (à calculer au Notebook 5)

On n'a **pas encore** calculé `score_final`. Ça se fait proprement dans le
**Notebook 3bis** avec la fonction de coût multiplicative complète intégrant
tes règles métiers (exclusion des nationales, hiérarchie des pénalités, etc.).

**Arrête ici le Notebook 3**, ouvre le Notebook 3bis pour le scoring.

In [ ]:
# Rapport final du Notebook 3
with engine.connect() as conn:
    final = pd.read_sql(text("""
        SELECT
            (SELECT COUNT(*) FROM osm_edges) AS total_edges,
            (SELECT COUNT(*) FROM osm_edges WHERE geom IS NOT NULL) AS edges_with_geom,
            (SELECT COUNT(*) FROM osm_edges WHERE has_bnac) AS edges_bnac,
            (SELECT COUNT(*) FROM osm_edges WHERE node_density IS NOT NULL) AS with_density,
            (SELECT COUNT(*) FROM osm_edges WHERE greenery_ratio > 0) AS with_greenery,
            (SELECT COUNT(*) FROM strava_segments) AS strava_segs
    """), conn)
print("=" * 60)
print("FIN NOTEBOOK 3 — DONNÉES PRÊTES POUR SCORING")
print("=" * 60)
for k, v in final.iloc[0].items():
    print(f"  {k:25s} : {int(v):,}")
print("=" * 60)
print("\n→ Passer au Notebook 3bis pour le scoring.")

FIN NOTEBOOK 3 — DONNÉES PRÊTES POUR SCORING
  total_edges               : 831,949
  edges_with_geom           : 831,892
  edges_bnac                : 285,313
  with_density              : 831,949
  with_greenery             : 662,609
  strava_segs               : 6,130

→ Passer au Notebook 3bis pour le scoring.


trop de segment vert ? il faudra peut etre affiner les selections.

demander si les arretes avec lequel on a fait c'est tout l'ile de france ou juste les segments selectionné ???

Autre question, j'ai vu qu'avec les arretes si on les traces bah ca refais pas le parcours d'un gpx classique du club, mais du coup si je vais à un point A d'un point B et que un moment je prend la trace d'un club il va le suivre a la lettre ou approximativement ?

Un gros soucis c'est aussi les routes de trop mauvaise qualité, on est en 25mm il faut vraiment bannir les routes gravel quoi